# 5. Policy Gradient Methods and Actor-Critic Algorithms

## 5.1 Introduction to Policy Gradient Methods

Policy Gradient (PG) methods represent a fundamental class of reinforcement learning algorithms that directly optimize the policy rather than learning value functions. This approach is particularly valuable for topology optimization where continuous action spaces and complex decision-making processes are common.

### 5.1.1 From Value-Based to Policy-Based Methods

Value-based methods like DQN learn a value function $Q(s,a)$ and derive policy implicitly:
$$\pi(a|s) = \arg\max_a Q(s,a)$$

Policy-based methods directly parameterize the policy $\pi_\theta(a|s)$ and optimize parameters $\theta$:
$$\theta^* = \arg\max_\theta J(\theta)$$

Where $J(\theta)$ is the expected return:
$$J(\theta) = \mathbb{E}_{\pi_\theta}[\sum_{t=0}^T \gamma^t r_t]$$

### 5.1.2 Advantages of Policy Gradient Methods

**1. Continuous Action Spaces:**
- Naturally handle continuous actions without discretization
- Essential for fine-grained topology optimization
- Avoid curse of dimensionality in action space

**2. Stochastic Policies:**
- Learn probability distributions over actions
- Enable exploration through policy randomness
- Better for partially observable environments

**3. Convergence Properties:**
- Guaranteed convergence to local optimum
- Avoid oscillations common in value-based methods
- More stable learning dynamics

**4. Policy Incorporation:**
- Can incorporate prior knowledge directly into policy
- Natural way to handle constraints
- Easier to shape desired behaviors

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import Categorical, Normal

class PolicyNetwork(nn.Module):
    """Neural network policy for topology optimization"""
    
    def __init__(self, state_dim, action_dim, hidden_dims=[128, 64], continuous=False):
        super(PolicyNetwork, self).__init__()
        
        self.continuous = continuous
        self.action_dim = action_dim
        
        # Build hidden layers
        layers = []
        prev_dim = state_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            prev_dim = hidden_dim
        
        self.feature_extractor = nn.Sequential(*layers)
        
        if continuous:
            # Continuous actions: mean and std
            self.mean_head = nn.Linear(prev_dim, action_dim)
            self.std_head = nn.Linear(prev_dim, action_dim)
        else:
            # Discrete actions: logits
            self.action_head = nn.Linear(prev_dim, action_dim)
    
    def forward(self, state):
        features = self.feature_extractor(state)
        
        if self.continuous:
            # Continuous action distribution
            mean = self.mean_head(features)
            std = F.softplus(self.std_head(features)) + 1e-5
            return mean, std
        else:
            # Discrete action distribution
            action_logits = self.action_head(features)
            return action_logits

class REINFORCEAgent:
    """REINFORCE (Monte Carlo Policy Gradient) Agent"""
    
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, continuous=False):
        self.policy = PolicyNetwork(state_dim, action_dim, continuous=continuous)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.gamma = gamma
        self.continuous = continuous
        
        # For training
        self.log_probs = []
        self.rewards = []
        
    def select_action(self, state):
        """Select action according to current policy"""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        
        if self.continuous:
            mean, std = self.policy(state_tensor)
            dist = Normal(mean, std)
            action = dist.sample()
            log_prob = dist.log_prob(action).sum(dim=-1)
            action = action.squeeze(0).detach().numpy()
        else:
            action_logits = self.policy(state_tensor)
            dist = Categorical(logits=action_logits)
            action = dist.sample()
            log_prob = dist.log_prob(action)
            action = action.item()
        
        # Store log probability for training
        self.log_probs.append(log_prob)
        
        return action
    
    def store_reward(self, reward):
        """Store reward for current timestep"""
        self.rewards.append(reward)
    
    def compute_returns(self):
        """Compute discounted returns for episode"""
        returns = []
        R = 0
        
        for reward in reversed(self.rewards):
            R = reward + self.gamma * R
            returns.insert(0, R)
        
        returns = torch.FloatTensor(returns)
        
        # Normalize returns for stability
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        return returns
    
    def update_policy(self):
        """Update policy using REINFORCE algorithm"""
        if len(self.log_probs) == 0 or len(self.rewards) == 0:
            return
        
        # Compute discounted returns
        returns = self.compute_returns()
        
        # Compute policy gradient loss
        policy_loss = []
        
        for log_prob, R in zip(self.log_probs, returns):
            policy_loss.append(-log_prob * R)
        
        policy_loss = torch.stack(policy_loss).sum()
        
        # Update policy
        self.optimizer.zero_grad()
        policy_loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        
        self.optimizer.step()
        
        # Clear episode data
        self.log_probs = []
        self.rewards = []
        
        return policy_loss.item()

# Demonstrate REINFORCE algorithm
def demonstrate_reinforce():
    """Demonstrate REINFORCE algorithm on a simple problem"""
    
    print("REINFORCE Algorithm Demonstration")
    print("="*40)
    
    # Create agent
    state_dim = 10
    action_dim = 5
    agent = REINFORCEAgent(state_dim, action_dim, lr=1e-3, continuous=False)
    
    print(f"State dimension: {state_dim}")
    print(f"Action dimension: {action_dim}")
    print(f"Policy parameters: {sum(p.numel() for p in agent.policy.parameters()):,}")
    
    # Simulate training episodes
    losses = []
    episode_rewards = []
    
    for episode in range(100):
        state = np.random.randn(state_dim)
        total_reward = 0
        
        # Generate episode
        for step in range(10):
            action = agent.select_action(state)
            # Simple reward: higher action = higher reward
            reward = action * 0.1 + np.random.randn() * 0.05
            agent.store_reward(reward)
            total_reward += reward
            state = state + np.random.randn(state_dim) * 0.1  # State transition
        
        # Update policy
        loss = agent.update_policy()
        
        if loss is not None:
            losses.append(loss)
            episode_rewards.append(total_reward)
        
        if episode % 20 == 0:
            print(f"Episode {episode:3d}: Reward = {total_reward:6.2f}, Loss = {loss if loss else 0:.4f}")
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    ax1.plot(episode_rewards)
    ax1.set_title('Episode Rewards')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(losses)
    ax2.set_title('Policy Loss')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Loss')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return agent

agent = demonstrate_reinforce()

## 5.2 Actor-Critic Methods

Actor-Critic methods combine the strengths of value-based and policy-based approaches by maintaining both a policy (actor) and a value function (critic). This combination addresses the high variance issues in pure policy gradient methods.

### 5.2.1 Actor-Critic Framework

The Actor-Critic framework consists of two components:

**Actor (Policy):** $\pi_\theta(a|s)$
- Selects actions based on current policy
- Updated using policy gradient methods
- Objective: maximize expected return

**Critic (Value Function):** $V_w(s)$ or $Q_w(s,a)$
- Evaluates the quality of states or state-action pairs
- Provides lower-variance gradient estimates
- Updated using temporal difference methods

### 5.2.2 Advantage Function

The advantage function measures how much better an action is compared to the average action:

$$A(s,a) = Q(s,a) - V(s)$$

Using advantages in policy gradients reduces variance:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}[\nabla_\theta \log \pi_\theta(a|s) \cdot A(s,a)]$$

### 5.2.3 Actor-Critic Algorithm Variants

**1. A2C (Advantage Actor-Critic):**
- Synchronous, deterministic version of A3C
- Uses multiple parallel environments
- Computes advantage estimates using bootstrapping
- Stable and widely used in practice

**2. PPO (Proximal Policy Optimization):**
- Uses clipped surrogate objective
- Prevents large policy updates
- Combines benefits of trust region methods
- State-of-the-art performance on many tasks

**3. SAC (Soft Actor-Critic):**
- Maximum entropy framework
- Off-policy algorithm
- Stable and sample-efficient
- Excellent for continuous control

In [ ]:
class ValueNetwork(nn.Module):
    """Value function network for critic"""
    
    def __init__(self, state_dim, hidden_dims=[128, 64]):
        super(ValueNetwork, self).__init__()
        
        layers = []
        prev_dim = state_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
    
    def forward(self, state):
        return self.network(state)

class A2CAgent:
    """Advantage Actor-Critic Agent"""
    
    def __init__(self, state_dim, action_dim, lr_actor=1e-3, lr_critic=1e-3, 
                 gamma=0.99, value_loss_coef=0.5, entropy_coef=0.01, continuous=False):
        
        self.gamma = gamma
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.continuous = continuous
        
        # Actor and Critic networks
        self.actor = PolicyNetwork(state_dim, action_dim, continuous=continuous)
        self.critic = ValueNetwork(state_dim)
        
        # Optimizers
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # Training storage
        self.log_probs = []
        self.values = []
        self.rewards = []
        self.entropies = []
        
    def select_action(self, state):
        """Select action and compute value estimate"""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        
        # Get value estimate
        value = self.critic(state_tensor)
        
        # Get action distribution
        if self.continuous:
            mean, std = self.actor(state_tensor)
            dist = Normal(mean, std)
            action = dist.sample()
            log_prob = dist.log_prob(action).sum(dim=-1)
            entropy = dist.entropy().sum(dim=-1)
            action = action.squeeze(0).detach().numpy()
        else:
            action_logits = self.actor(state_tensor)
            dist = Categorical(logits=action_logits)
            action = dist.sample()
            log_prob = dist.log_prob(action)
            entropy = dist.entropy()
            action = action.item()
        
        # Store training data
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.entropies.append(entropy)
        
        return action
    
    def store_reward(self, reward):
        """Store reward for current timestep"""
        self.rewards.append(reward)
    
    def compute_returns_and_advantages(self):
        """Compute returns and advantages"""
        returns = []
        advantages = []
        
        # Compute discounted returns
        R = 0
        for reward, value in zip(reversed(self.rewards), reversed(self.values)):
            R = reward + self.gamma * R
            returns.insert(0, R)
        
        returns = torch.tensor(returns, dtype=torch.float32)
        values = torch.cat(self.values).squeeze()
        
        # Compute advantages
        advantages = returns - values
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return returns, advantages
    
    def update(self):
        """Update actor and critic networks"""
        if len(self.log_probs) == 0:
            return
        
        # Compute returns and advantages
        returns, advantages = self.compute_returns_and_advantages()
        
        # Compute actor loss
        actor_loss = []
        for log_prob, advantage, entropy in zip(self.log_probs, advantages, self.entropies):
            actor_loss.append(-log_prob * advantage - self.entropy_coef * entropy)
        
        actor_loss = torch.stack(actor_loss).sum()
        
        # Compute critic loss
        values = torch.cat(self.values).squeeze()
        critic_loss = F.mse_loss(values, returns)
        
        # Update actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward(retain_graph=True)
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
        self.actor_optimizer.step()
        
        # Update critic
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_optimizer.step()
        
        # Clear episode data
        self.log_probs = []
        self.values = []
        self.rewards = []
        self.entropies = []
        
        return actor_loss.item(), critic_loss.item()

# Demonstrate A2C algorithm
def demonstrate_a2c():
    """Demonstrate A2C algorithm"""
    
    print("A2C Algorithm Demonstration")
    print("="*35)
    
    # Create agent
    state_dim = 10
    action_dim = 4
    agent = A2CAgent(state_dim, action_dim, continuous=False)
    
    print(f"State dimension: {state_dim}")
    print(f"Action dimension: {action_dim}")
    print(f"Actor parameters: {sum(p.numel() for p in agent.actor.parameters()):,}")
    print(f"Critic parameters: {sum(p.numel() for p in agent.critic.parameters()):,}")
    
    # Training metrics
    actor_losses = []
    critic_losses = []
    episode_rewards = []
    
    # Simulate training
    for episode in range(200):
        state = np.random.randn(state_dim)
        total_reward = 0
        
        # Generate episode
        for step in range(15):
            action = agent.select_action(state)
            # Reward that depends on action quality
            reward = action * 0.2 + np.random.randn() * 0.1
            agent.store_reward(reward)
            total_reward += reward
            state = state + np.random.randn(state_dim) * 0.05
        
        # Update networks
        actor_loss, critic_loss = agent.update()
        
        if actor_loss is not None:
            actor_losses.append(actor_loss)
            critic_losses.append(critic_loss)
            episode_rewards.append(total_reward)
        
        if episode % 40 == 0:
            print(f"Episode {episode:3d}: Reward = {total_reward:6.2f}, "
                  f"Actor Loss = {actor_loss:.4f}, Critic Loss = {critic_loss:.4f}")
    
    # Plot training progress
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    axes[0, 0].plot(episode_rewards)
    axes[0, 0].set_title('Episode Rewards')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(actor_losses, label='Actor', color='blue')
    axes[0, 1].plot(critic_losses, label='Critic', color='red')
    axes[0, 1].set_title('Training Losses')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Moving average of rewards
    if len(episode_rewards) >= 10:
        moving_avg = np.convolve(episode_rewards, np.ones(10)/10, mode='valid')
        axes[1, 0].plot(moving_avg)
        axes[1, 0].set_title('Moving Average Reward (10 episodes)')
        axes[1, 0].set_xlabel('Episode')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Loss ratio
    if len(actor_losses) > 0 and len(critic_losses) > 0:
        loss_ratio = np.array(actor_losses) / (np.array(critic_losses) + 1e-8)
        axes[1, 1].plot(loss_ratio)
        axes[1, 1].set_title('Actor/Critic Loss Ratio')
        axes[1, 1].set_xlabel('Episode')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return agent

a2c_agent = demonstrate_a2c()

## 5.3 Advanced Policy Gradient Techniques

This section explores advanced techniques that improve the stability, efficiency, and performance of policy gradient methods for topology optimization.

### 5.3.1 Proximal Policy Optimization (PPO)

PPO addresses the policy update problem by constraining the policy change to prevent destructive updates:

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left(r_t(\theta)A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)A_t\right) \right]$$

Where:
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the probability ratio
- $A_t$ is the advantage estimate
- $\epsilon$ is the clipping parameter (typically 0.1 or 0.2)

### 5.3.2 Trust Region Policy Optimization (TRPO)

TRPO constrains policy updates using a trust region defined by KL-divergence:

$$\max_\theta \mathbb{E}_t \left[ \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)} A_t \right]$$

Subject to:
$$\mathbb{E}_t \left[ KL\left(\pi_{\theta_{old}}(\cdot|s_t) \parallel \pi_\theta(\cdot|s_t)\right) \right] \leq \delta$$

### 5.3.3 Generalized Advantage Estimation (GAE)

GAE provides a trade-off between bias and variance in advantage estimation:

$$A_t^{GAE(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

Where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD residual.

### 5.3.4 Natural Policy Gradient

Natural policy gradient uses the Fisher information matrix to ensure consistent update scales:

$$\theta_{k+1} = \theta_k + \alpha F^{-1} \nabla_\theta J(\theta_k)$$

Where $F$ is the Fisher information matrix:
$$F = \mathbb{E}_{\pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) \nabla_\theta \log \pi_\theta(a|s)^T \right]$$

In [ ]:
class PPOAgent:
    """Proximal Policy Optimization Agent"""
    
    def __init__(self, state_dim, action_dim, lr=3e-4, gamma=0.99, 
                 gae_lambda=0.95, clip_epsilon=0.2, value_loss_coef=0.5, 
                 entropy_coef=0.01, ppo_epochs=4, mini_batch_size=64, continuous=False):
        
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.ppo_epochs = ppo_epochs
        self.mini_batch_size = mini_batch_size
        self.continuous = continuous
        
        # Networks
        self.actor = PolicyNetwork(state_dim, action_dim, continuous=continuous)
        self.critic = ValueNetwork(state_dim)
        
        # Optimizers
        self.optimizer = optim.Adam(list(self.actor.parameters()) + list(self.critic.parameters()), lr=lr)
        
        # Storage for trajectory data
        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        self.values = []
        self.dones = []
    
    def select_action(self, state):
        """Select action and store training data"""
        state_tensor = torch.FloatTensor(state)
        
        with torch.no_grad():
            value = self.critic(state_tensor)
            
            if self.continuous:
                mean, std = self.actor(state_tensor)
                dist = Normal(mean, std)
                action = dist.sample()
                log_prob = dist.log_prob(action).sum(dim=-1)
                action = action.numpy()
            else:
                action_logits = self.actor(state_tensor)
                dist = Categorical(logits=action_logits)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                action = action.item()
        
        # Store data
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.values.append(value)
        
        return action
    
    def store_reward(self, reward, done):
        """Store reward and done flag"""
        self.rewards.append(reward)
        self.dones.append(done)
    
    def compute_gae(self):
        """Compute Generalized Advantage Estimation"""
        advantages = []
        gae = 0
        
        for i in reversed(range(len(self.rewards))):
            if i == len(self.rewards) - 1:
                next_value = 0
            else:
                next_value = self.values[i + 1].item()
            
            value = self.values[i].item()
            reward = self.rewards[i]
            done = self.dones[i]
            
            delta = reward + self.gamma * next_value * (1 - done) - value
            gae = delta + self.gamma * self.gae_lambda * (1 - done) * gae
            advantages.insert(0, gae)
        
        advantages = torch.tensor(advantages, dtype=torch.float32)
        returns = advantages + torch.cat(self.values).squeeze()
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return returns, advantages
    
    def update(self):
        """Update policy using PPO algorithm"""
        if len(self.states) == 0:
            return
        
        # Compute GAE advantages
        returns, advantages = self.compute_gae()
        
        # Convert to tensors
        states = torch.FloatTensor(np.array(self.states))
        actions = torch.LongTensor(self.actions) if not self.continuous else torch.FloatTensor(np.array(self.actions))
        old_log_probs = torch.stack(self.log_probs).squeeze()
        old_values = torch.cat(self.values).squeeze()
        
        # PPO update
        total_actor_loss = 0
        total_critic_loss = 0
        total_entropy_loss = 0
        
        for _ in range(self.ppo_epochs):
            # Generate random minibatches
            indices = torch.randperm(len(states))
            
            for start in range(0, len(states), self.mini_batch_size):
                end = start + self.mini_batch_size
                batch_indices = indices[start:end]
                
                batch_states = states[batch_indices]
                batch_actions = actions[batch_indices]
                batch_old_log_probs = old_log_probs[batch_indices]
                batch_old_values = old_values[batch_indices]
                batch_advantages = advantages[batch_indices]
                batch_returns = returns[batch_indices]
                
                # Get current policy and value predictions
                values = self.critic(batch_states).squeeze()
                
                if self.continuous:
                    mean, std = self.actor(batch_states)
                    dist = Normal(mean, std)
                    log_probs = dist.log_prob(batch_actions).sum(dim=-1)
                    entropy = dist.entropy().sum(dim=-1).mean()
                else:
                    action_logits = self.actor(batch_states)
                    dist = Categorical(logits=action_logits)
                    log_probs = dist.log_prob(batch_actions)
                    entropy = dist.entropy().mean()
                
                # Compute probability ratio
                ratio = torch.exp(log_probs - batch_old_log_probs)
                
                # Compute clipped surrogate loss
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * batch_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                
                # Compute value loss
                critic_loss = F.mse_loss(values, batch_returns)
                
                # Total loss
                loss = actor_loss + self.value_loss_coef * critic_loss - self.entropy_coef * entropy
                
                # Update networks
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
                torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
                self.optimizer.step()
                
                total_actor_loss += actor_loss.item()
                total_critic_loss += critic_loss.item()
                total_entropy_loss += entropy.item()
        
        # Clear trajectory data
        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        self.values = []
        self.dones = []
        
        n_updates = self.ppo_epochs * (len(states) // self.mini_batch_size + 1)
        
        return (total_actor_loss / n_updates, 
                total_critic_loss / n_updates, 
                total_entropy_loss / n_updates)

# Compare different policy gradient methods
def compare_pg_methods():
    """Compare REINFORCE, A2C, and PPO algorithms"""
    
    print("Policy Gradient Methods Comparison")
    print("="*40)
    
    state_dim = 8
    action_dim = 4
    n_episodes = 100
    
    # Create agents
    reinforce_agent = REINFORCEAgent(state_dim, action_dim)
    a2c_agent = A2CAgent(state_dim, action_dim)
    ppo_agent = PPOAgent(state_dim, action_dim)
    
    agents = {
        'REINFORCE': reinforce_agent,
        'A2C': a2c_agent,
        'PPO': ppo_agent
    }
    
    results = {name: [] for name in agents.keys()}
    
    for episode in range(n_episodes):
        for name, agent in agents.items():
            state = np.random.randn(state_dim)
            total_reward = 0
            
            # Generate episode
            for step in range(10):
                action = agent.select_action(state)
                reward = action * 0.15 + np.random.randn() * 0.05
                
                if hasattr(agent, 'store_reward'):
                    agent.store_reward(reward)
                else:
                    agent.store_reward(reward, step == 9)  # PPO needs done flag
                
                total_reward += reward
                state = state + np.random.randn(state_dim) * 0.1
            
            # Update policy
            if name == 'REINFORCE':
                loss = agent.update_policy()
            elif name == 'A2C':
                actor_loss, critic_loss = agent.update()
            else:  # PPO
                actor_loss, critic_loss, entropy_loss = agent.update()
            
            results[name].append(total_reward)
    
    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Learning curves
    for name, rewards in results.items():
        if len(rewards) >= 10:
            moving_avg = np.convolve(rewards, np.ones(10)/10, mode='valid')
            ax1.plot(moving_avg, label=name, linewidth=2)
    
    ax1.set_title('Learning Curves Comparison (10-episode moving average)')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Average Reward')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Final performance comparison
    final_performance = {name: np.mean(rewards[-20:]) for name, rewards in results.items()}
    bars = ax2.bar(range(len(final_performance)), list(final_performance.values()))
    ax2.set_xticks(range(len(final_performance)))
    ax2.set_xticklabels(list(final_performance.keys()))
    ax2.set_ylabel('Final Performance (20-episode avg)')
    ax2.set_title('Final Performance Comparison')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, value in zip(bars, final_performance.values()):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                  f'{value:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\nPerformance Summary:")
    for name, performance in final_performance.items():
        std_dev = np.std(results[name][-20:])
        print(f"{name:12}: {performance:.3f} ± {std_dev:.3f}")
    
    return results

comparison_results = compare_pg_methods()

## 5.4 Policy Gradient for Topology Optimization

This section adapts policy gradient methods specifically for topology optimization problems, addressing the unique challenges and requirements of electromagnetic design.

### 5.4.1 Action Space Design

Topology optimization requires careful consideration of the action space to ensure both effectiveness and feasibility.

**Continuous Action Spaces:**
- **Material density actions**: $a \in [0, 1]$ for each cell representing material density
- **Geometric parameters**: Curvature control, boundary smoothness
- **Field manipulation**: Control variables for boundary conditions

**Discrete Action Spaces:**
- **Cell-based actions**: Add/remove material at specific locations
- **Region-based actions**: Modify predefined regions simultaneously
- **Pattern actions**: Apply design patterns or templates

**Hybrid Approaches:**
- **Hierarchical actions**: High-level region selection + low-level material assignment
- **Multi-step actions**: Sequential design operations
- **Conditional actions**: Actions depend on current design state

### 5.4.2 State Representation for Policy Networks

Effective state representation is crucial for policy gradient success in topology optimization:

**Multi-Channel Representation:**
- Material distribution channels
Physical field channels (magnetic, electric, thermal)
Gradient and derivative information
Constraint violation indicators

**Feature Engineering:**
- Local material density statistics
Regional performance metrics
Boundary and connectivity information
Manufacturing constraint indicators

**Hierarchical Features:**
- Local cell-level features
Regional patch-level features
Global design-level features
Multi-scale information fusion

In [ ]:
class TopologyPolicyGradient:
    """Policy Gradient agent specialized for topology optimization"""
    
    def __init__(self, grid_size=20, action_type='continuous', n_regions=8):
        self.grid_size = grid_size
        self.action_type = action_type
        self.n_regions = n_regions
        
        # Define action space
        if action_type == 'continuous':
            # Continuous: material density for each cell
            self.action_dim = grid_size * grid_size
        elif action_type == 'discrete':
            # Discrete: add/remove material per cell
            self.action_dim = grid_size * grid_size * 2  # Each cell: add/remove
        elif action_type == 'regional':
            # Regional: modify predefined regions
            self.action_dim = n_regions * 2  # Each region: add/remove
        
        # State dimension (multi-channel)
        self.state_dim = self._compute_state_dim()
        
        # Create policy network
        self.policy = PolicyNetwork(
            self.state_dim, 
            self.action_dim, 
            hidden_dims=[256, 128, 64],
            continuous=(action_type == 'continuous')
        )
        
        # Create value network
        self.value_network = ValueNetwork(self.state_dim, hidden_dims=[256, 128])
        
        # Optimizer
        self.optimizer = optim.Adam(
            list(self.policy.parameters()) + list(self.value_network.parameters()),
            lr=3e-4
        )
        
        # Training parameters
        self.gamma = 0.99
        self.gae_lambda = 0.95
        self.clip_epsilon = 0.2
        self.entropy_coef = 0.01
        self.value_loss_coef = 0.5
        
        # Experience storage
        self.reset_storage()
        
        # Region definitions for regional actions
        self.regions = self._create_regions()
    
    def _compute_state_dim(self):
        """Compute state dimension based on multi-channel representation"""
        # Material channels: current design, gradient, connectivity
        material_channels = 3 * self.grid_size * self.grid_size
        
        # Physical field channels: magnetic field, force density
        physics_channels = 2 * self.grid_size * self.grid_size
        
        # Constraint channels: material limits, manufacturing constraints
        constraint_channels = 2 * self.grid_size * self.grid_size
        
        # Global features: performance metrics, statistics
        global_features = 20
        
        return material_channels + physics_channels + constraint_channels + global_features
    
    def _create_regions(self):
        """Create predefined regions for regional actions"""
        regions = []
        
        # Divide grid into regions
        region_size = self.grid_size // int(np.sqrt(self.n_regions))
        
        for i in range(0, self.grid_size, region_size):
            for j in range(0, self.grid_size, region_size):
                region = {
                    'start': (i, j),
                    'end': (min(i + region_size, self.grid_size), 
                           min(j + region_size, self.grid_size))
                }
                regions.append(region)
                
                if len(regions) >= self.n_regions:
                    break
            
            if len(regions) >= self.n_regions:
                break
        
        return regions[:self.n_regions]
    
    def _extract_state_features(self, design, fields, constraints):
        """Extract multi-channel state features"""
        features = []
        
        # Flatten spatial features
        features.extend(design.flatten())
        
        # Add gradient information
        grad_y, grad_x = np.gradient(design)
        features.extend(grad_y.flatten())
        features.extend(grad_x.flatten())
        
        # Add connectivity information (simplified)
        from scipy import ndimage
        labeled, num_components = ndimage.label(design > 0.5)
        connectivity = (labeled > 0).astype(float)
        features.extend(connectivity.flatten())
        
        # Add physics fields
        for field_name, field_data in fields.items():
            features.extend(field_data.flatten())
        
        # Add constraint information
        for constraint_name, constraint_data in constraints.items():
            features.extend(constraint_data.flatten())
        
        # Add global features
        features.append(np.mean(design))
        features.append(np.std(design))
        features.append(np.sum(design) / design.size)
        features.append(num_components)
        
        # Add field statistics
        for field_data in fields.values():
            features.append(np.mean(field_data))
            features.append(np.std(field_data))
            features.append(np.max(field_data))
        
        # Add constraint statistics
        for constraint_data in constraints.values():
            features.append(np.mean(constraint_data))
            features.append(np.max(constraint_data))
        
        return np.array(features)
    
    def select_action(self, state_features):
        """Select action based on current policy"""
        state_tensor = torch.FloatTensor(state_features).unsqueeze(0)
        
        with torch.no_grad():
            if self.action_type == 'continuous':
                mean, std = self.policy(state_tensor)
                dist = Normal(mean, std)
                action = dist.sample()
                log_prob = dist.log_prob(action).sum(dim=-1)
                action = action.squeeze(0).numpy()
                
                # Clip actions to valid range [0, 1]
                action = np.clip(action, 0, 1)
                
            else:
                action_logits = self.policy(state_tensor)
                dist = Categorical(logits=action_logits)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                action = action.item()
        
        return action, log_prob.item()
    
    def apply_action(self, action, design):
        """Apply action to modify design"""
        new_design = design.copy()
        
        if self.action_type == 'continuous':
            # Continuous: update material densities
            new_design = action.reshape(self.grid_size, self.grid_size)
            
        elif self.action_type == 'discrete':
            # Discrete: cell-based add/remove
            cell_idx = action // 2
            operation = action % 2  # 0: remove, 1: add
            
            row = cell_idx // self.grid_size
            col = cell_idx % self.grid_size
            
            if operation == 0:
                new_design[row, col] = 0
            else:
                new_design[row, col] = 1
                
        elif self.action_type == 'regional':
            # Regional: modify predefined regions
            region_idx = action // 2
            operation = action % 2
            
            if region_idx < len(self.regions):
                region = self.regions[region_idx]
                start_y, start_x = region['start']
                end_y, end_x = region['end']
                
                if operation == 0:
                    new_design[start_y:end_y, start_x:end_x] = 0
                else:
                    new_design[start_y:end_y, start_x:end_x] = 1
        
        return new_design
    
    def reset_storage(self):
        """Reset experience storage"""
        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        self.values = []
        self.dones = []
    
    def store_transition(self, state, action, reward, log_prob, value, done):
        """Store transition data"""
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.dones.append(done)
    
    def update_policy(self, ppo_epochs=4, mini_batch_size=64):
        """Update policy using PPO-style update"""
        if len(self.states) == 0:
            return
        
        # Compute GAE advantages
        advantages = self._compute_gae()
        returns = advantages + torch.cat(self.values).squeeze()
        
        # Convert to tensors
        states = torch.FloatTensor(np.array(self.states))
        actions = torch.LongTensor(self.actions)
        old_log_probs = torch.FloatTensor(self.log_probs)
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        total_loss = 0
        
        for _ in range(ppo_epochs):
            indices = torch.randperm(len(states))
            
            for start in range(0, len(states), mini_batch_size):
                end = start + mini_batch_size
                batch_indices = indices[start:end]
                
                batch_states = states[batch_indices]
                batch_actions = actions[batch_indices]
                batch_old_log_probs = old_log_probs[batch_indices]
                batch_advantages = advantages[batch_indices]
                batch_returns = returns[batch_indices]
                
                # Get current predictions
                values = self.value_network(batch_states).squeeze()
                
                if self.action_type == 'continuous':
                    mean, std = self.policy(batch_states)
                    dist = Normal(mean, std)
                    log_probs = dist.log_prob(batch_actions).sum(dim=-1)
                    entropy = dist.entropy().sum(dim=-1).mean()
                else:
                    action_logits = self.policy(batch_states)
                    dist = Categorical(logits=action_logits)
                    log_probs = dist.log_prob(batch_actions)
                    entropy = dist.entropy().mean()
                
                # Compute losses
                ratio = torch.exp(log_probs - batch_old_log_probs)
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * batch_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                
                critic_loss = F.mse_loss(values, batch_returns)
                
                loss = actor_loss + self.value_loss_coef * critic_loss - self.entropy_coef * entropy
                
                # Update
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                torch.nn.utils.clip_grad_norm_(self.value_network.parameters(), 0.5)
                self.optimizer.step()
                
                total_loss += loss.item()
        
        # Clear storage
        self.reset_storage()
        
        return total_loss / (ppo_epochs * (len(states) // mini_batch_size + 1))
    
    def _compute_gae(self):
        """Compute Generalized Advantage Estimation"""
        advantages = []
        gae = 0
        
        for i in reversed(range(len(self.rewards))):
            if i == len(self.rewards) - 1:
                next_value = 0
            else:
                next_value = self.values[i + 1].item()
            
            value = self.values[i].item()
            reward = self.rewards[i]
            done = self.dones[i]
            
            delta = reward + self.gamma * next_value * (1 - done) - value
            gae = delta + self.gamma * self.gae_lambda * (1 - done) * gae
            advantages.insert(0, gae)
        
        return torch.tensor(advantages, dtype=torch.float32)

# Demonstrate topology policy gradient agent
def demonstrate_topology_pg():
    """Demonstrate specialized topology optimization policy gradient"""
    
    print("Topology Optimization Policy Gradient Agent")
    print("="*45)
    
    # Test different action types
    action_types = ['continuous', 'discrete', 'regional']
    
    for action_type in action_types:
        print(f"\nTesting {action_type} action type:")
        print("-" * 30)
        
        agent = TopologyPolicyGradient(grid_size=12, action_type=action_type)
        
        print(f"Grid size: {agent.grid_size}x{agent.grid_size}")
        print(f"State dimension: {agent.state_dim}")
        print(f"Action dimension: {agent.action_dim}")
        print(f"Policy parameters: {sum(p.numel() for p in agent.policy.parameters()):,}")
        
        # Create sample data
        design = np.random.random((agent.grid_size, agent.grid_size))
        fields = {
            'magnetic': np.random.random((agent.grid_size, agent.grid_size)),
            'force': np.random.random((agent.grid_size, agent.grid_size))
        }
        constraints = {
            'material_limit': np.ones((agent.grid_size, agent.grid_size)),
            'manufacturing': np.zeros((agent.grid_size, agent.grid_size))
        }
        
        # Extract state features
        state_features = agent._extract_state_features(design, fields, constraints)
        
        # Select action
        action, log_prob = agent.select_action(state_features)
        
        # Apply action
        new_design = agent.apply_action(action, design)
        
        print(f"Action: {action}")
        print(f"Log probability: {log_prob:.4f}")
        print(f"Design change: {np.sum(np.abs(new_design - design)):.2f}")
        
        # Visualize before/after
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        ax1.imshow(design, cmap='binary')
        ax1.set_title('Original Design')
        ax1.axis('off')
        
        ax2.imshow(new_design, cmap='binary')
        ax2.set_title(f'After {action_type} Action')
        ax2.axis('off')
        
        plt.suptitle(f'{action_type.capitalize()} Action Type Demonstration')
        plt.show()

demonstrate_topology_pg()

## 5.5 Training Strategies for Topology Optimization

Effective training of policy gradient methods for topology optimization requires specialized strategies that account for the unique characteristics of electromagnetic design problems.

### 5.5.1 Curriculum Learning

Curriculum learning gradually increases problem complexity to improve training stability and convergence:

**Progressive Grid Refinement:**
- Start with coarse grids (8×8 or 16×16)
- Progressively refine to finer grids (32×32, 64×64)
- Transfer learned policies between resolutions
- Reduces computational cost in early training

**Simplified Physics to Complex Physics:**
- Begin with simplified electromagnetic models
- Gradually introduce nonlinear material properties
- Add multi-physics coupling progressively
- Include manufacturing constraints gradually

**Single-Objective to Multi-Objective:**
- Start with single objective optimization (force maximization)
- Introduce secondary objectives (material minimization)
- Add constraint handling progressively
- Balance competing objectives through reward shaping

### 5.5.2 Reward Shaping and Design

Effective reward design is crucial for guiding policy learning in topology optimization:

**Multi-Component Reward Function:**

$$r_t = w_1 \cdot r_{\text{performance}} + w_2 \cdot r_{\text{material}} + w_3 \cdot r_{\text{constraints}} + w_4 \cdot r_{\text{smoothness}}$$

Where:
- $r_{\text{performance}}$: Force/torque improvement
- $r_{\text{material}}$: Material usage efficiency
- $r_{\text{constraints}}$: Constraint satisfaction/penalty
- $r_{\text{smoothness}}$: Design smoothness and manufacturability

**Potential-Based Reward Shaping:**

$$r'_t = r_t + \gamma \Phi(s_{t+1}) - \Phi(s_t)$$

Where $\Phi(s)$ is a potential function that guides learning without changing optimal policy.

### 5.5.3 Exploration Strategies

Effective exploration is essential for discovering good topology designs:

**Entropy Regularization:**
- Add entropy bonus to encourage exploration
- Balance exploration vs. exploitation through coefficient tuning
- Anneal entropy coefficient during training

**Noisy Networks:**
- Add parameter noise to policy network
- Enables structured exploration
- Reduces need for high-entropy policies

**Curiosity-Driven Exploration:**
- Intrinsic reward based on prediction error
- Encourages visiting novel states
- Particularly useful for sparse reward environments

In [ ]:
class TopologyRewardShaper:
    """Reward shaping for topology optimization"""
    
    def __init__(self, weights=None):
        # Default weights for different reward components
        self.weights = weights or {
            'performance': 1.0,
            'material': -0.3,
            'constraints': -2.0,
            'smoothness': -0.1,
            'connectivity': -0.2
        }
        
        # History for incremental rewards
        self.prev_performance = None
        self.prev_material = None
        
    def compute_reward(self, design, performance, fields, constraints):
        """Compute shaped reward for current design"""
        
        # Performance reward (force/torque improvement)
        performance_reward = self._compute_performance_reward(performance)
        
        # Material efficiency reward
        material_reward = self._compute_material_reward(design)
        
        # Constraint satisfaction penalty
        constraint_reward = self._compute_constraint_reward(design, constraints)
        
        # Smoothness reward
        smoothness_reward = self._compute_smoothness_reward(design)
        
        # Connectivity reward
        connectivity_reward = self._compute_connectivity_reward(design)
        
        # Combine rewards
        total_reward = (
            self.weights['performance'] * performance_reward +
            self.weights['material'] * material_reward +
            self.weights['constraints'] * constraint_reward +
            self.weights['smoothness'] * smoothness_reward +
            self.weights['connectivity'] * connectivity_reward
        )
        
        # Update history
        self.prev_performance = performance
        self.prev_material = np.sum(design) / design.size
        
        return total_reward, {
            'performance': performance_reward,
            'material': material_reward,
            'constraints': constraint_reward,
            'smoothness': smoothness_reward,
            'connectivity': connectivity_reward
        }
    
    def _compute_performance_reward(self, performance):
        """Compute performance-based reward"""
        if self.prev_performance is None:
            # First step, reward absolute performance
            return performance['force'] * 0.1
        else:
            # Reward improvement over previous performance
            improvement = performance['force'] - self.prev_performance['force']
            return improvement * 10.0  # Scale up improvements
    
    def _compute_material_reward(self, design):
        """Compute material efficiency reward"""
        current_material = np.sum(design) / design.size
        
        if self.prev_material is None:
            return 0.0
        else:
            # Penalize material increase
            material_change = current_material - self.prev_material
            return material_change * 5.0
    
    def _compute_constraint_reward(self, design, constraints):
        """Compute constraint satisfaction reward"""
        penalty = 0.0
        
        # Material limit constraint
        max_material_ratio = constraints.get('max_material_ratio', 0.4)
        current_material = np.sum(design) / design.size
        
        if current_material > max_material_ratio:
            penalty += (current_material - max_material_ratio) * 10.0
        
        # Manufacturing constraint (minimum feature size)
        min_feature_size = constraints.get('min_feature_size', 2)
        feature_violations = self._count_feature_violations(design, min_feature_size)
        penalty += feature_violations * 0.5
        
        return -penalty
    
    def _compute_smoothness_reward(self, design):
        """Compute design smoothness reward"""
        # Compute gradient magnitude
        grad_y, grad_x = np.gradient(design)
        gradient_mag = np.sqrt(grad_x**2 + grad_y**2)
        
        # Penalize high gradients (encourage smooth designs)
        smoothness_penalty = np.mean(gradient_mag)
        
        return -smoothness_penalty
    
    def _compute_connectivity_reward(self, design):
        """Compute connectivity reward"""
        from scipy import ndimage
        
        # Count connected components
        labeled, num_components = ndimage.label(design > 0.5)
        
        # Penalize disconnected designs (prefer single connected component)
        connectivity_penalty = max(0, num_components - 1)
        
        return -connectivity_penalty
    
    def _count_feature_violations(self, design, min_feature_size):
        """Count violations of minimum feature size constraint"""
        from scipy import ndimage
        
        # Find connected components
        labeled, num_features = ndimage.label(design > 0.5)
        
        violations = 0
        for i in range(1, num_features + 1):
            feature_size = np.sum(labeled == i)
            if feature_size < min_feature_size**2:
                violations += 1
        
        return violations

class CurriculumTrainer:
    """Curriculum learning trainer for topology optimization"""
    
    def __init__(self, base_agent):
        self.agent = base_agent
        self.reward_shaper = TopologyRewardShaper()
        self.current_stage = 0
        self.stages = self._define_curriculum()
        
    def _define_curriculum(self):
        """Define curriculum stages"""
        return [
            {
                'name': 'Coarse Grid - Simple Physics',
                'grid_size': 8,
                'physics_complexity': 'simple',
                'objectives': ['performance'],
                'episodes': 200,
                'reward_weights': {'performance': 1.0, 'material': 0.0}
            },
            {
                'name': 'Medium Grid - Material Constraint',
                'grid_size': 16,
                'physics_complexity': 'simple',
                'objectives': ['performance', 'material'],
                'episodes': 300,
                'reward_weights': {'performance': 1.0, 'material': -0.3}
            },
            {
                'name': 'Fine Grid - Full Physics',
                'grid_size': 32,
                'physics_complexity': 'full',
                'objectives': ['performance', 'material', 'constraints'],
                'episodes': 500,
                'reward_weights': {'performance': 1.0, 'material': -0.3, 'constraints': -2.0}
            }
        ]
    
    def advance_stage(self):
        """Advance to next curriculum stage"""
        if self.current_stage < len(self.stages) - 1:
            self.current_stage += 1
            stage = self.stages[self.current_stage]
            
            # Update reward weights
            self.reward_shaper.weights.update(stage['reward_weights'])
            
            print(f"\nAdvancing to Stage {self.current_stage + 1}: {stage['name']}")
            print(f"Grid size: {stage['grid_size']}")
            print(f"Objectives: {', '.join(stage['objectives'])}")
            
            return True
        return False
    
    def train_stage(self, n_episodes):
        """Train for current curriculum stage"""
        stage = self.stages[self.current_stage]
        
        print(f"\nTraining Stage {self.current_stage + 1}: {stage['name']}")
        print(f"Episodes: {n_episodes}")
        print(f"Grid size: {stage['grid_size']}")
        
        episode_rewards = []
        
        for episode in range(n_episodes):
            # Simulate training episode
            total_reward = self._simulate_training_episode(stage)
            episode_rewards.append(total_reward)
            
            if episode % 50 == 0:
                avg_reward = np.mean(episode_rewards[-50:]) if len(episode_rewards) >= 50 else total_reward
                print(f"Episode {episode:3d}: Average Reward = {avg_reward:6.2f}")
            
            # Check for stage completion
            if len(episode_rewards) >= 100:
                recent_avg = np.mean(episode_rewards[-50:])
                if recent_avg > 0.5:  # Performance threshold
                    print(f"Stage completed at episode {episode}")
                    break
        
        return episode_rewards
    
    def _simulate_training_episode(self, stage):
        """Simulate a training episode for current stage"""
        # This is a simplified simulation
        # In practice, this would interact with the actual environment
        
        base_reward = np.random.exponential(0.5)
        
        # Adjust reward based on stage difficulty
        stage_difficulty = (self.current_stage + 1) * 0.3
        adjusted_reward = base_reward - stage_difficulty
        
        # Add some noise
        noise = np.random.normal(0, 0.2)
        
        return max(adjusted_reward + noise, -1.0)
    
    def train_curriculum(self):
        """Train through complete curriculum"""
        print("Starting Curriculum Learning for Topology Optimization")
        print("="*60)
        
        all_rewards = []
        
        for stage_idx, stage in enumerate(self.stages):
            stage_rewards = self.train_stage(stage['episodes'])
            all_rewards.extend(stage_rewards)
            
            if stage_idx < len(self.stages) - 1:
                self.advance_stage()
        
        # Plot learning progress across curriculum
        plt.figure(figsize=(12, 6))
        
        # Plot overall learning curve
        plt.plot(all_rewards, alpha=0.7, label='Episode Rewards')
        
        # Mark stage boundaries
        episode_count = 0
        for i, stage in enumerate(self.stages):
            if i < len(self.stages) - 1:
                episode_count += len(all_rewards) // len(self.stages)
                plt.axvline(x=episode_count, color='red', linestyle='--', alpha=0.7)
                plt.text(episode_count + 5, max(all_rewards) * 0.9, 
                        f"Stage {i+1} → {i+2}", rotation=0)
        
        plt.xlabel('Episode')
        plt.ylabel('Reward')
        plt.title('Curriculum Learning Progress')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
        
        return all_rewards

# Demonstrate curriculum learning
def demonstrate_curriculum_learning():
    """Demonstrate curriculum learning approach"""
    
    print("Curriculum Learning Demonstration")
    print("="*35)
    
    # Create a base agent
    base_agent = TopologyPolicyGradient(grid_size=16, action_type='continuous')
    
    # Create curriculum trainer
    trainer = CurriculumTrainer(base_agent)
    
    # Show curriculum stages
    print("\nCurriculum Stages:")
    for i, stage in enumerate(trainer.stages):
        print(f"\nStage {i+1}: {stage['name']}")
        print(f"  Grid Size: {stage['grid_size']}")
        print(f"  Objectives: {', '.join(stage['objectives'])}")
        print(f"  Episodes: {stage['episodes']}")
        print(f"  Reward Weights: {stage['reward_weights']}")
    
    # Train curriculum
    rewards = trainer.train_curriculum()
    
    print(f"\nCurriculum Training Completed!")
    print(f"Total episodes: {len(rewards)}")
    print(f"Final performance: {np.mean(rewards[-50:]):.3f}")
    
    return trainer

# Run demonstrations
curriculum_trainer = demonstrate_curriculum_learning()

## 5.6 Comparative Analysis and Best Practices

This section provides a comprehensive comparison of different policy gradient methods and offers practical guidance for their application to topology optimization.

### 5.6.1 Algorithm Comparison Summary

| Algorithm | Action Space | Sample Efficiency | Stability | Implementation Complexity | Best For |
|-----------|--------------|-------------------|-----------|-------------------------|----------|
| REINFORCE | Discrete/Continuous | Low | Medium | Low | Simple problems, educational purposes |
| A2C | Discrete/Continuous | Medium | High | Medium | General purpose, balanced performance |
| PPO | Discrete/Continuous | High | Very High | High | Complex problems, state-of-the-art |
| TRPO | Discrete/Continuous | High | Very High | Very High | High-stakes applications, theoretical guarantees |
| SAC | Continuous | Very High | High | High | Continuous control, sample efficiency critical |

### 5.6.2 Performance Metrics for Topology Optimization

**Design Quality Metrics:**
- **Objective Achievement**: Force/torque improvement percentage
- **Material Efficiency**: Performance per unit material
- **Constraint Satisfaction**: Violation frequency and severity
- **Manufacturability**: Feature size compliance, connectivity

**Training Efficiency Metrics:**
- **Convergence Speed**: Episodes to reach 90% of final performance
- **Sample Efficiency**: Number of FEM evaluations required
- **Stability**: Performance variance across runs
- **Computational Cost**: Training time and memory usage

**Robustness Metrics:**
- **Generalization**: Performance on unseen problems
- **Hyperparameter Sensitivity**: Performance variation with parameter changes
- **Scalability**: Performance with problem size increase
- **Reproducibility**: Consistency across random seeds

### 5.6.3 Best Practices for Topology Optimization

**1. Algorithm Selection Guidelines:**
- Start with A2C for balanced performance and stability
- Use PPO for complex problems requiring high performance
- Consider SAC for continuous control problems
- Use REINFORCE only for simple educational purposes

**2. State Representation Best Practices:**
- Use multi-channel representations for rich information
- Include both local and global features
- Normalize features to consistent scales
- Incorporate physics-based features when available

**3. Action Space Design Guidelines:**
- Choose action space based on problem requirements
- Continuous actions for fine-grained control
- Discrete actions for discrete design decisions
- Regional actions for large-scale modifications

**4. Training Strategy Recommendations:**
- Use curriculum learning for complex problems
- Implement proper reward shaping
- Monitor multiple performance metrics
- Use early stopping to prevent overfitting

**5. Hyperparameter Tuning Guidelines:**
- Start with conservative learning rates (1e-4 to 1e-3)
- Use appropriate entropy coefficients (0.01 to 0.1)
- Set discount factors based on problem horizon (0.95 to 0.99)
- Tune value/policy loss coefficients (0.5 to 2.0)

In [ ]:
# Comprehensive comparison and analysis
def comprehensive_policy_gradient_analysis():
    """Comprehensive analysis of policy gradient methods for topology optimization"""
    
    print("Comprehensive Policy Gradient Analysis for Topology Optimization")
    print("="*70)
    
    # Create comparison data
    algorithms = ['REINFORCE', 'A2C', 'PPO', 'TRPO', 'SAC']
    
    comparison_metrics = {
        'Convergence Speed': [4, 7, 9, 8, 9],
        'Sample Efficiency': [3, 6, 8, 8, 10],
        'Stability': [5, 8, 9, 10, 8],
        'Implementation Complexity': [2, 5, 8, 10, 7],
        'Scalability': [4, 7, 9, 8, 9],
        'Robustness': [5, 7, 9, 9, 8]
    }
    
    # Create radar chart comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Radar chart
    categories = list(comparison_metrics.keys())
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    ax1.set_theta_offset(np.pi / 2)
    ax1.set_theta_direction(-1)
    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(categories)
    ax1.set_ylim(0, 10)
    ax1.set_title('Policy Gradient Methods Comparison\n(Radar Chart)', size=14, fontweight='bold')
    ax1.grid(True)
    
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    
    for i, algo in enumerate(algorithms):
        values = [comparison_metrics[metric][i] for metric in categories]
        values += values[:1]
        
        ax1.plot(angles, values, 'o-', linewidth=2, label=algo, color=colors[i])
        ax1.fill(angles, values, alpha=0.1, color=colors[i])
    
    ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    
    # Bar chart for specific metrics
    metrics_to_show = ['Convergence Speed', 'Sample Efficiency', 'Stability']
    x = np.arange(len(algorithms))
    width = 0.25
    
    for i, metric in enumerate(metrics_to_show):
        values = [comparison_metrics[metric][j] for j in range(len(algorithms))]
        ax2.bar(x + i*width, values, width, label=metric, alpha=0.8, color=colors[i])
    
    ax2.set_xlabel('Algorithm')
    ax2.set_ylabel('Score (1-10)')
    ax2.set_title('Key Performance Metrics Comparison')
    ax2.set_xticks(x + width)
    ax2.set_xticklabels(algorithms, rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim(0, 11)
    
    plt.tight_layout()
    plt.show()
    
    # Create detailed comparison table
    detailed_comparison = {
        'Algorithm': algorithms,
        'Best For': [
            'Simple problems, educational use',
            'General purpose, balanced performance',
            'Complex problems, state-of-the-art',
            'High-stakes applications',
            'Continuous control, sample efficiency'
        ],
        'Pros': [
            'Simple implementation, clear theory',
            'Good balance, stable learning',
            'High performance, widely used',
            'Theoretical guarantees',
            'Sample efficient, off-policy'
        ],
        'Cons': [
            'High variance, slow learning',
            'Moderate sample efficiency',
            'Complex implementation',
            'Very complex, computationally expensive',
            'Requires careful tuning'
        ],
        'Recommended For TO': [
            'Educational purposes only',
            'Good starting point',
            'Recommended for complex problems',
            'When theoretical guarantees needed',
            'Continuous action spaces'
        ]
    }
    
    df_comparison = pd.DataFrame(detailed_comparison)
    
    print("\nDetailed Algorithm Comparison:")
    print("="*50)
    print(df_comparison.to_string(index=False))
    
    return df_comparison

# Create implementation checklist
def create_pg_implementation_checklist():
    """Create implementation checklist for policy gradient methods"""
    
    checklist = {
        'Problem Formulation': [
            'Define clear action space (discrete/continuous)',
            'Design multi-channel state representation',
            'Create balanced reward function',
            'Specify termination conditions',
            'Identify manufacturing constraints'
        ],
        'Algorithm Selection': [
            'Assess problem complexity',
            'Consider action space type',
            'Evaluate sample efficiency needs',
            'Check computational resources',
            'Review implementation expertise'
        ],
        'Network Architecture': [
            'Choose appropriate network size',
            'Select activation functions',
            'Add regularization techniques',
            'Initialize weights properly',
            'Test forward and backward passes'
        ],
        'Training Configuration': [
            'Set learning rates for actor/critic',
            'Configure entropy coefficient',
            'Define GAE parameters',
            'Set PPO clipping parameters',
            'Configure training schedule'
        ],
        'Reward Shaping': [
            'Balance multiple objectives',
            'Design constraint penalties',
            'Add intermediate rewards',
            'Test reward function sensitivity',
            'Validate reward doesn\'t change optimal policy'
        ],
        'Training Strategy': [
            'Implement curriculum learning',
            'Use proper exploration strategies',
            'Monitor multiple metrics',
            'Set up early stopping',
            'Plan hyperparameter tuning'
        ],
        'Evaluation': [
            'Define evaluation protocols',
            'Track design quality metrics',
            'Perform ablation studies',
            'Validate with physics simulations',
            'Test generalization capabilities'
        ]
    }
    
    print("\nPolicy Gradient Implementation Checklist")
    print("="*50)
    
    for category, items in checklist.items():
        print(f"\n{category}:")
        print("-" * (len(category) + 1))
        for i, item in enumerate(items, 1):
            print(f"  {i}. {item}")
    
    return checklist

# Run comprehensive analysis
comparison_df = comprehensive_policy_gradient_analysis()
checklist = create_pg_implementation_checklist()

print("\n" + "="*80)
print("POLICY GRADIENT METHODS FOR TOPOLOGY OPTIMIZATION - KEY INSIGHTS")
print("="*80)
print("\n1. Algorithm Recommendations:")
print("   - Start with A2C for balanced performance and implementation simplicity")
print("   - Use PPO for complex problems requiring high performance")
print("   - Consider SAC for continuous control with sample efficiency requirements")
print("   - Avoid REINFORCE except for educational purposes")

print("\n2. Topology Optimization Specific Considerations:")
print("   - Use multi-channel state representations for physics-rich information")
print("   - Design action spaces that respect manufacturing constraints")
print("   - Implement curriculum learning for complex design problems")
print("   - Balance multiple competing objectives through reward shaping")

print("\n3. Best Practices:")
print("   - Monitor both design quality and training efficiency")
print("   - Use entropy regularization for effective exploration")
print("   - Validate final designs with physics simulations")
print("   - Test generalization across different design scenarios")

print("\n4. Common Pitfalls to Avoid:")
print("   - Poorly designed reward functions leading to local optima")
print("   - Inadequate exploration causing premature convergence")
print("   - Ignoring manufacturing constraints in action space")
print("   - Focusing only on training efficiency ignoring design quality")